In [1]:
import pandas as pd
import os
import glob
from pathlib import Path

# === 1. CẤU HÌNH ===
data_dir = '../../processed/clean_data'
output_dir = '../../processed/KPI_requirements'
os.makedirs(output_dir, exist_ok=True)
file_pattern = os.path.join(data_dir, '*.parquet')

# === 2. HÀM TÍNH KPI ===
def calculate_kpi(group):
    trips = len(group)
    if trips == 0:
        return pd.Series({
            'trips': 0, 'duration_p50': 0, 'duration_p95': 0, 'speed_p50': 0,
            'revenue_total': 0, 'tip_percent_median': 0,
            'flex_fare_percent': 0, 'airport_percent': 0, 'invalid_percent': 0
        })

    return pd.Series({
        'trips': trips,
        'duration_p50': group['trip_duration'].median(),
        'duration_p95': group['trip_duration'].quantile(0.95),
        'speed_p50': group['speed_mph'].median(),
        'revenue_total': group['total_amount'].sum(),
        'tip_percent_median': (group['tip_amount'] / group['total_amount'] * 100).median(),
        'flex_fare_percent': group['is_flex_fare'].sum() / trips * 100,
        'airport_percent': (group['airport_fee'] > 0).sum() / trips * 100,
        'invalid_percent': 0.0
    })

# === 3. XỬ LÝ TUẦN TỰ ===
daily_results = []
weekly_results = [] 
monthly_results = []
list_files = sorted(glob.glob(file_pattern))
print(f"Tìm thấy {len(list_files)} files.")

for file_path in list_files:
    file_name = os.path.basename(file_path)
    try:
        df_chunk = pd.read_parquet(file_path)
        original_count = len(df_chunk)

        if 'valid_all' in df_chunk.columns:
            df_chunk = df_chunk[df_chunk['valid_all'] == 1].copy()

        if df_chunk.empty:
            print(f"File {file_name}: 0/{original_count} dòng hợp lệ. Bỏ qua.")
            continue

        print(f"Xử lý {file_name}: {len(df_chunk)} dòng OK.")

        df_chunk['tpep_pickup_datetime'] = pd.to_datetime(df_chunk['tpep_pickup_datetime'])
        df_chunk['airport_fee'] = df_chunk['airport_fee'].fillna(0) if 'airport_fee' in df_chunk.columns else 0
        if 'is_flex_fare' not in df_chunk.columns:
            df_chunk['is_flex_fare'] = 0

        # Tạo các trục thời gian chuyên dụng
        df_chunk['pickup_date'] = df_chunk['tpep_pickup_datetime'].dt.date
        df_chunk['pickup_week'] = df_chunk['tpep_pickup_datetime'].dt.to_period('W').apply(lambda r: r.start_time.date())
        df_chunk['pickup_month'] = df_chunk['tpep_pickup_datetime'].dt.to_period('M')

        # Tính toán phân đoạn (Tránh lỗi include_groups)
        daily_chunk = df_chunk.groupby('pickup_date').apply(calculate_kpi, include_groups=False).reset_index()
        daily_results.append(daily_chunk)

        weekly_chunk = df_chunk.groupby('pickup_week').apply(calculate_kpi, include_groups=False).reset_index()
        weekly_results.append(weekly_chunk)

        monthly_chunk = df_chunk.groupby('pickup_month').apply(calculate_kpi, include_groups=False).reset_index()
        monthly_results.append(monthly_chunk)

    except Exception as e:
        print(f"Lỗi xử lý file {file_name}: {e}")
    
    if 'df_chunk' in locals():
        del df_chunk

# === 5. GỘP, KHỬ TRÙNG VÀ LƯU KẾT QUẢ ===
if daily_results:
    print("\n🔄 Đang tiến hành gộp và tối ưu hóa dữ liệu KPI cuối cùng...")

    # --- 1. DAILY (Gộp lại nếu trùng ngày giữa các file) ---
    final_daily = pd.concat(daily_results, ignore_index=True)
    # Hàm xử lý gom nhóm lần cuối phòng trường hợp trùng lặp mốc thời gian giữa các file
    final_daily = final_daily.groupby('pickup_date').agg({
        'trips': 'sum',
        'duration_p50': 'median',
        'duration_p95': 'max',
        'speed_p50': 'median',
        'revenue_total': 'sum',
        'tip_percent_median': 'median',
        'flex_fare_percent': 'mean',
        'airport_percent': 'mean',
        'invalid_percent': 'mean'
    }).reset_index().sort_values('pickup_date')
    
    final_daily = final_daily[final_daily["trips"] > 100]
    final_daily['trips_index'] = 100 * final_daily['trips'] / final_daily['trips'].max()
    final_daily.to_csv(f'{output_dir}/kpi_daily_2021.csv', index=False)
    print("✅ Đã lưu Daily KPI thành công!")

    # --- 2. WEEKLY (Khử trùng lặp tuần giao thoa giữa các file) ---
    final_weekly = pd.concat(weekly_results, ignore_index=True)
    final_weekly = final_weekly.groupby('pickup_week').agg({
        'trips': 'sum',
        'duration_p50': 'median',
        'duration_p95': 'max',
        'speed_p50': 'median',
        'revenue_total': 'sum',
        'tip_percent_median': 'median',
        'flex_fare_percent': 'mean',
        'airport_percent': 'mean',
        'invalid_percent': 'mean'
    }).reset_index().sort_values('pickup_week')
    
    final_weekly = final_weekly[final_weekly['trips'] > 100]
    final_weekly['trips_index'] = 100 * final_weekly['trips'] / final_weekly['trips'].max()
    final_weekly.to_csv(f'{output_dir}/kpi_weekly_2021.csv', index=False)
    print("✅ Đã lưu Weekly KPI thành công!")

    # --- 3. MONTHLY (Khử trùng lặp tháng) ---
    final_monthly = pd.concat(monthly_results, ignore_index=True)
    final_monthly = final_monthly.groupby('pickup_month').agg({
        'trips': 'sum',
        'duration_p50': 'median',
        'duration_p95': 'max',
        'speed_p50': 'median',
        'revenue_total': 'sum',
        'tip_percent_median': 'median',
        'flex_fare_percent': 'mean',
        'airport_percent': 'mean',
        'invalid_percent': 'mean'
    }).reset_index().sort_values('pickup_month')
    
    final_monthly = final_monthly[final_monthly['trips'] > 1000]
    final_monthly['pickup_month'] = final_monthly['pickup_month'].dt.to_timestamp().dt.date
    final_monthly['trips_index'] = 100 * final_monthly['trips'] / final_monthly['trips'].max()
    final_monthly['percent_total_year'] = 100 * final_monthly['trips'] / final_monthly['trips'].sum()
    final_monthly.to_csv(f'{output_dir}/kpi_monthly_2021.csv', index=False)
    print("✅ Đã lưu Monthly KPI thành công!")

else:
    print("❌ Không có dữ liệu nào hợp lệ để xử lý.")

Tìm thấy 12 files.
Xử lý clean_2021-01.parquet: 1216266 dòng OK.
Xử lý clean_2021-02.parquet: 1220238 dòng OK.
Xử lý clean_2021-03.parquet: 1721706 dòng OK.
Xử lý clean_2021-04.parquet: 1956905 dòng OK.
Xử lý clean_2021-05.parquet: 2283514 dòng OK.
Xử lý clean_2021-06.parquet: 2594701 dòng OK.
Xử lý clean_2021-07.parquet: 2572559 dòng OK.
Xử lý clean_2021-08.parquet: 2531123 dòng OK.
Xử lý clean_2021-09.parquet: 2682406 dòng OK.
Xử lý clean_2021-10.parquet: 3184813 dòng OK.
Xử lý clean_2021-11.parquet: 3205510 dòng OK.
Xử lý clean_2021-12.parquet: 2980000 dòng OK.

🔄 Đang tiến hành gộp và tối ưu hóa dữ liệu KPI cuối cùng...
✅ Đã lưu Daily KPI thành công!
✅ Đã lưu Weekly KPI thành công!
✅ Đã lưu Monthly KPI thành công!
